# Exploration de FrozenLake

## Objectif

Comprendre le fonctionnement de FrozenLake et les principales méthodes
de l’API Gymnasium avant d’implémenter un agent.

## Environnement étudié

La carte comporte 16 cases, numérotées de 0 à 15 :
- `S` : départ.
- `F` : glace praticable.
- `H` : trou.
- `G` : objectif.

L’agent dispose de quatre actions :
- `0` : gauche.
- `1` : bas.
- `2` : droite.
- `3` : haut.

## API utilisée

- `reset()` démarre un épisode et renvoie `(state, info)`.
- `step(action)` renvoie
  `(state, reward, terminated, truncated, info)`.
- `render()` affiche l’environnement.
- `close()` libère ses ressources.

Les expériences comparent les déplacements sans glissade et avec glissade.

In [1]:
import sys

print(sys.executable)

c:\Users\Utilisateur\Documents\GitHub\frozenlake-rl\.venv\Scripts\python.exe


In [2]:
import gymnasium as gym

env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="ansi",
)

state, info = env.reset(seed=42)

print("État initial :", state)
print(env.render())

État initial : 0

SFFF
FHFH
FFFH
HFFG



In [3]:
state, info = env.reset(seed=42)

state, reward, terminated, truncated, info = env.step(1)

print("État :", state)
print("Récompense :", reward)
print("Terminé :", terminated)
print("Interrompu par une limite :", truncated)
print("Informations :", info)
print(env.render())

État : 4
Récompense : 0
Terminé : False
Interrompu par une limite : False
Informations : {'prob': 1.0}
  (Down)
SFFF
FHFH
FFFH
HFFG



In [4]:
actions = [1, 1, 2, 2, 1, 2]

state, info = env.reset(seed=42)

for action in actions:
    state, reward, terminated, truncated, info = env.step(action)

    print(
        f"Action : {action} | État : {state} | "
        f"Récompense : {reward} | Terminé : {terminated}"
    )

    if terminated or truncated:
        break

print(env.render())

Action : 1 | État : 4 | Récompense : 0 | Terminé : False
Action : 1 | État : 8 | Récompense : 0 | Terminé : False
Action : 2 | État : 9 | Récompense : 0 | Terminé : False
Action : 2 | État : 10 | Récompense : 0 | Terminé : False
Action : 1 | État : 14 | Récompense : 0 | Terminé : False
Action : 2 | État : 15 | Récompense : 1 | Terminé : True
  (Right)
SFFF
FHFH
FFFH
HFFG



## Parcours manuel sans glissade

Avec `is_slippery=False`, chaque action produit le déplacement demandé,
sauf si un bord de la carte le bloque.

La séquence d’actions `[1, 1, 2, 2, 1, 2]` suit le chemin :
`0 → 4 → 8 → 9 → 10 → 14 → 15`.

L’agent atteint l’objectif en six actions.
Les déplacements intermédiaires donnent une récompense de `0`.
À l’arrivée, la récompense vaut `1` et `terminated` vaut `True`.

Ce parcours est choisi manuellement : aucun apprentissage n’a encore lieu.

In [5]:
print("Espace d’observation :", env.observation_space)
print("Nombre d’états :", env.observation_space.n)

print("Espace d’action :", env.action_space)
print("Nombre d’actions :", env.action_space.n)

Espace d’observation : Discrete(16)
Nombre d’états : 16
Espace d’action : Discrete(4)
Nombre d’actions : 4


In [6]:
state, info = env.reset(seed=42)

for action in [1, 2]:
    state, reward, terminated, truncated, info = env.step(action)

    print(
        f"État : {state} | Récompense : {reward} | "
        f"Terminé : {terminated}"
    )

    if terminated or truncated:
        break

print(env.render())

État : 4 | Récompense : 0 | Terminé : False
État : 5 | Récompense : 0 | Terminé : True
  (Right)
SFFF
FHFH
FFFH
HFFG



## Fin d’un épisode

`terminated=True` indique la fin de l’épisode, qu’il s’agisse
d’une réussite ou d’un échec.

- Objectif atteint : `reward=1`, `terminated=True`.
- Trou atteint : `reward=0`, `terminated=True`.

Après la fin d’un épisode, il faut appeler `reset()` avant de jouer à nouveau.

In [7]:
limited_env = gym.make(
    "FrozenLake-v1",
    is_slippery=False,
    max_episode_steps=3,
)

state, info = limited_env.reset(seed=42)

for step in range(1, 4):
    state, reward, terminated, truncated, info = limited_env.step(0)

    print(
        f"Étape : {step} | État : {state} | "
        f"Terminé : {terminated} | Limite atteinte : {truncated}"
    )

    if terminated or truncated:
        break

limited_env.close()

Étape : 1 | État : 0 | Terminé : False | Limite atteinte : False
Étape : 2 | État : 0 | Terminé : False | Limite atteinte : False
Étape : 3 | État : 0 | Terminé : False | Limite atteinte : True


## Arrêt par une limite

`terminated` indique une fin liée aux règles du jeu :
l’agent atteint un trou ou l’objectif.

`truncated` indique un arrêt imposé par une limite extérieure,
comme le nombre maximal d’actions.

Dans l’expérience limitée à trois actions, l’agent reste au départ.
À la troisième action :
- `terminated=False`
- `truncated=True`

L’épisode est fini dès que `terminated or truncated` vaut `True`.

In [8]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="ansi",
)

state, info = slippery_env.reset(seed=42)

for action in actions:
    previous_state = state

    state, reward, terminated, truncated, info = slippery_env.step(action)

    print(
        f"Départ : {previous_state} | Action : {action} | "
        f"Arrivée : {state} | Récompense : {reward} | "
        f"Terminé : {terminated}"
    )

    if terminated or truncated:
        break

print(slippery_env.render())
slippery_env.close()

Départ : 0 | Action : 1 | Arrivée : 4 | Récompense : 0 | Terminé : False
Départ : 4 | Action : 1 | Arrivée : 5 | Récompense : 0 | Terminé : True
  (Down)
SFFF
FHFH
FFFH
HFFG



## Parcours sur la glace glissante

Avec `is_slippery=True`, le déplacement réel peut différer de l’action demandée.

En rejouant le parcours manuel avec `seed=42`, l’agent suit :
`0 → 4 → 5`.

La deuxième action demande de descendre vers l’état `8`,
mais l’agent glisse à droite vers le trou à l’état `5`.

L’épisode se termine avec `reward=0` et `terminated=True`.
Un chemin qui réussit sans glissade ne garantit donc pas la réussite
sur la glace glissante.

In [9]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="ansi",
)

state, info = slippery_env.reset(seed=7)

for action in actions:
    previous_state = state

    state, reward, terminated, truncated, info = slippery_env.step(action)

    print(
        f"Départ : {previous_state} | Action : {action} | "
        f"Arrivée : {state} | Récompense : {reward} | "
        f"Terminé : {terminated}"
    )

    if terminated or truncated:
        break

print(slippery_env.render())
slippery_env.close()

Départ : 0 | Action : 1 | Arrivée : 1 | Récompense : 0 | Terminé : False
Départ : 1 | Action : 1 | Arrivée : 2 | Récompense : 0 | Terminé : False
Départ : 2 | Action : 2 | Arrivée : 6 | Récompense : 0 | Terminé : False
Départ : 6 | Action : 2 | Arrivée : 10 | Récompense : 0 | Terminé : False
Départ : 10 | Action : 1 | Arrivée : 11 | Récompense : 0 | Terminé : True
  (Down)
SFFF
FHFH
FFFH
HFFG



## Reproductibilité et influence de la graine

Avec les mêmes actions :
- `seed=42` produit le parcours `0 → 4 → 5`.
- `seed=7` produit le parcours `0 → 1 → 2 → 6 → 10 → 11`.

Relancer une expérience avec la même graine et les mêmes actions
reproduit le même parcours dans les mêmes conditions.

Changer la graine peut modifier les glissades et donc les cases visitées.